# Amazon Bedrock AgentCore Code Interpreter에서 명령 실행 - 튜토리얼

이 튜토리얼에서는 Amazon Bedrock AgentCore Code Interpreter를 사용하여 명령(셸 및 AWS CLI)을 실행하는 방법을 알아봅니다. AWS 서비스와 상호 작용하며, 특히 S3 작업을 중점적으로 다룹니다. 다음 단계를 진행합니다.

1. Code Interpreter 생성
2. Code Interpreter 세션 시작
3. 명령 실행(셸 및 AWS CLI)
5. S3 작업 수행(버킷 생성, 객체 복사, 버킷 객체 나열)
6. 정리(세션 중지 및 Code Interpreter 삭제)



## 사전 요구 사항
- Bedrock AgentCore Code Interpreter에 액세스할 수 있는 AWS 계정
- Code Interpreter 리소스를 생성하고 관리하는 데 필요한 IAM 권한
- S3 작업을 수행하는 데 필요한 IAM 권한
- 필수 Python 패키지 설치(boto3 및 bedrock-agentcore 포함)


## IAM 실행 역할에 다음 IAM 정책을 연결해야 합니다

~~~ {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:CreateCodeInterpreter",
                "bedrock-agentcore:StartCodeInterpreterSession",
                "bedrock-agentcore:InvokeCodeInterpreter",
                "bedrock-agentcore:StopCodeInterpreterSession",
                "bedrock-agentcore:DeleteCodeInterpreter",
                "bedrock-agentcore:ListCodeInterpreters",
                "bedrock-agentcore:GetCodeInterpreter"
            ],
            "Resource": "*"
        },
        {
            "Effect": "Allow",
            "Action": [
                "logs:CreateLogGroup",
                "logs:CreateLogStream",
                "logs:PutLogEvents"
            ],
            "Resource": "arn:aws:logs:*:*:log-group:/aws/bedrock-agentcore/code-interpreter*"
        }
    ]
}

#### IAM 실행 역할에 다음 신뢰 정책도 연결해야 합니다. 신뢰 정책에 bedrock-agentcore.amazonaws.com을 포함하면 Bedrock Agent 서비스가 이 IAM 역할을 수임하여 사용자를 대신해 Amazon S3 등의 작업을 수행할 수 있습니다.

```{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "AWS": "arn:aws:iam::<account_id>:root",
                "Service": [
                    "bedrock-agentcore.amazonaws.com"
                ]
            },
            "Action": "sts:AssumeRole",
            "Condition": {}
        }
    ]
}

또한 이 튜토리얼에서 설명하는 S3 작업을 수행하려면 IAM 실행 역할에 AmazonS3FullAccess IAM 정책을 연결해야 합니다.

## 작동 방식

코드 실행 샌드박스는 Code Interpreter, 셸, 파일 시스템을 갖춘 격리 환경을 생성하여 에이전트가 사용자 질의를 안전하게 처리할 수 있도록 합니다. 대규모 언어 모델(LLM)이 도구 선택을 지원한 후 이 세션 내에서 코드가 실행되며, 결과는 종합을 위해 사용자 또는 에이전트에게 반환됩니다.

![로컬 아키텍처](code-interpreter.png)

## 1. 환경 설정

먼저 필요한 라이브러리를 가져옵니다.

In [ ]:
!pip install --upgrade -r requirements.txt

In [ ]:
import json
import boto3
from bedrock_agentcore._utils import endpoints
import time
from typing import Dict, Any

## 2. 구성 변수

Code Interpreter와 S3 작업에 필요한 구성 변수를 설정합니다. 또한 Code Interpreter가 역할을 수임하여 다른 AWS 리소스에 액세스할 수 있도록 IAM 실행 역할을 전달합니다. 위에서 설명한 대로 이 역할에는 S3 권한이 필요합니다.

In [ ]:
# 구성 설정
execution_role_arn = "<execution-role-arn>"
unique_bucket_name = f"amzn-bucket-{int(time.time())}"
s3_path = f"s3://{unique_bucket_name}"

region = "us-west-2"

# Code Interpreter에 업로드한 다음 S3 버킷에 업로드할 로컬 파일
local_file = "samples/stats.py"

## 3. 엔드포인트 설정

boto3 클라이언트를 생성하려면 데이터 플레인과 컨트롤 플레인 엔드포인트를 모두 구성해야 합니다.

In [ ]:
# 엔드포인트 구성
data_plane_endpoint = endpoints.get_data_plane_endpoint(region)
control_plane_endpoint = endpoints.get_control_plane_endpoint(region)

## 4. AWS 클라이언트 생성

컨트롤 플레인과 데이터 플레인 작업에 사용할 boto3 클라이언트를 초기화합니다.

In [ ]:
# boto3 클라이언트 생성
cp_client = boto3.client("bedrock-agentcore-control", region_name=region, endpoint_url=control_plane_endpoint)

dp_client = boto3.client("bedrock-agentcore", region_name=region, endpoint_url=data_plane_endpoint)

## 5. Code Interpreter 생성

지정된 구성 파라미터로 Code Interpreter 인스턴스를 생성합니다.

Code Interpreter를 구성할 때 네트워크 설정(Sandbox, Public 또는 VPC), 종속성 구성, 보안 설정을 선택할 수 있습니다. 또한 Code Interpreter가 액세스할 수 있는 AWS 리소스를 정의하는 IAM 런타임 역할을 통해 권한을 설정할 수 있습니다.

In [ ]:
# Code Interpreter 생성
unique_name = f"s3InteractionEnv_{int(time.time())}"
interpreter_response = cp_client.create_code_interpreter(
    name=unique_name,
    description="Environment for S3 file operations",
    executionRoleArn=execution_role_arn,
    networkConfiguration={"networkMode": "PUBLIC"},
)
interpreter_id = interpreter_response["codeInterpreterId"]
print(f"Created interpreter: {interpreter_id}")

## 6. 세션 시작

코드를 실행할 Code Interpreter 세션을 생성합니다.

In [ ]:
# 세션 시작
session_response = dp_client.start_code_interpreter_session(
    codeInterpreterIdentifier=interpreter_id,
    name="s3InteractionSession",
    sessionTimeoutSeconds=900,
)
session_id = session_response["sessionId"]
print(f"Created session: {session_id}")

## 7. 도구 실행용 헬퍼 함수

Code Interpreter 도구 호출을 간소화하는 유틸리티 함수를 정의합니다.

In [ ]:
def call_tool(tool_name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
    response = dp_client.invoke_code_interpreter(
        codeInterpreterIdentifier=interpreter_id,
        sessionId=session_id,
        name=tool_name,
        arguments=arguments,
    )
    for event in response["stream"]:
        return json.dumps(event["result"], indent=2)

## 8. 코드 실행 테스트

### 8.1 간단한 Hello World 예제로 Code Interpreter를 테스트합니다.

In [ ]:
# 코드 실행 테스트
# S3 작업
print("executing shell command \n")
command_response = call_tool("executeCommand", {"command": "echo 'Hello World'"})
print(f"command result: {command_response}")

# 결과 파싱 및 표시
command_results = json.loads(command_response)
print(command_results["structuredContent"]["stdout"])

### 8.2 다음으로 PIP를 사용하여 샌드박스에 boto3를 설치합니다.

In [ ]:
# 코드 실행 테스트
# S3 작업
print("executing shell command \n")
command_response = call_tool("executeCommand", {"command": "pip install boto3"})

# 결과 파싱 및 표시
command_results = json.loads(command_response)
print(command_results["structuredContent"]["stdout"])

## 9. 명령 실행을 통한 파일 작업 및 S3 연동

#### 9.1 샌드박스에 로컬 파일 쓰기

In [ ]:
# 샌드박스에 파일 쓰기
print("Writing file to sandbox")
try:
    with open(local_file, "r", encoding="utf-8") as local_file_content:
        local_file_content = local_file_content.read()
except FileNotFoundError:
    print(f"Error: The file '{local_file}' was not found.")
except Exception as e:
    print(f"An error occurred: {e}")

files_to_create = [{"path": "stats.py", "text": local_file_content}]
write_files_response = call_tool("writeFiles", {"content": files_to_create})
print(f"write files result: {write_files_response}")

#### 9.2 Code Interpreter를 통해 S3 버킷 생성

In [ ]:
# S3 작업
print("\nCreating S3 bucket")
create_s3_response = call_tool("executeCommand", {"command": f"aws s3 mb {s3_path} --region {region}"})
print(f"create result: {create_s3_response}")

#### 9.3 Code Interpreter에서 명령을 실행하여 위에서 생성한 S3 버킷에 파일 업로드

In [ ]:
print("\nUploading file to S3")
upload_to_s3_response = call_tool("executeCommand", {"command": f"aws s3 cp {files_to_create[0]['path']} {s3_path}"})
print(f"upload result: {upload_to_s3_response}")

#### 9.4 Code Interpreter에서 명령을 실행하여 S3 버킷의 파일 나열

In [ ]:
print("\nListing files in S3")
list_s3_response = call_tool("executeCommand", {"command": f"aws s3 ls {s3_path}"})
print(f"list result: {list_s3_response}")

## 10. 정리

세션을 중지하고 Code Interpreter를 삭제하여 리소스를 정리합니다.

In [ ]:
# 정리
print("Cleaning up session and interpreter")
dp_client.stop_code_interpreter_session(codeInterpreterIdentifier=interpreter_id, sessionId=session_id)
print("Session stopped successfully")

cp_client.delete_code_interpreter(codeInterpreterId=interpreter_id)
print("Interpreter deleted successfully")